# CTR Review Prioritization — FlyRank Search Intelligence Capstone

**Question.** Can safe page/context features estimate last-30-day CTR more accurately than a position-tier-only baseline, and which aggregate cohorts merit review?

This notebook uses the public anonymized `FlyRank/internship-starter` dataset. It does not access the gated full warehouse. The raw CSV is downloaded to `/tmp`, never into this repository. Published results are aggregate-only; do not add hashed IDs, page URLs, client names, private queries, or raw exports.

The evaluation is contemporaneous decision support, not a future forecast or causal estimate. The starter data card does not give absolute dates for its latest/previous windows, so this analysis reports only the relative 30-day window definition.

Sources: [dataset card](https://huggingface.co/datasets/FlyRank/internship-starter) · [internship guide](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/GUIDE.md) · [FlyRank](https://flyrank.ai).

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

# The analysis script downloads the public CSV into /tmp, performs five-fold
# client-grouped validation, and writes aggregate-only metrics and charts.
!python work/analyze.py
!python work/build_paper.py

## Data and label

The public starter release contains 30,000 rows, 53 columns, and 32 pseudonymized client groups. The target is `100 × clicks_last_30d / impressions_last_30d`. Pages with fewer than 50 latest-window impressions are excluded from fitting, evaluation, and ranked cohorts. Previous-30-day and 90-day fields are available, but they are not used as outcome-derived predictors.

## Feature and leakage controls

Predictors are average position, search volume, competition, word count, content age, days since update, content type, main intent, competition level, age tier, freshness tier, and position tier. The model excludes client/content IDs, click/impression-derived rates, sessions and engagement metrics, trend fields, product flags, health scores, and provider/model metadata. Position is retained as a same-period context variable, so this is not a prospective prediction task.

## Baseline, model, and validation

The baseline is a training-fold, impression-weighted mean CTR by position tier (global training-fold mean fallback). The challenger is a random forest with 250 trees, minimum leaf size 12, one-hot categorical fields, training-fold imputation, and square-root impression weights capped at the training-fold 95th percentile. Five-fold GroupKFold ensures each pseudonymized client appears in only one test fold. The primary metric is pooled out-of-fold impression-weighted MAE in CTR percentage points; lower is better.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

results = json.loads(Path('work/results.json').read_text())
a = results['analysis']
print(f"Eligible pages: {results['dataset']['eligible_pages']:,} of {results['dataset']['row_count']:,}")
print(f"Position-tier baseline weighted MAE: {a['baseline_weighted_mae_pp']:.3f} percentage points")
print(f"Random-forest weighted MAE: {a['model_weighted_mae_pp']:.3f} percentage points")
print(f"Relative reduction: {a['mae_improvement_vs_baseline_pct']:.1f}% (modest and not uniform across folds)")
display(pd.DataFrame(a['fold_metrics']))
display(pd.DataFrame(a['ranked_aggregate_recommendations'])[[
    'content_type', 'main_intent', 'position_tier', 'eligible_pages',
    'observed_ctr_pct', 'expected_ctr_pct', 'positive_gap_click_proxy'
]].head(10))

In [ ]:
display(Image(filename='paper/assets/model-vs-baseline.png'))
display(Image(filename='paper/assets/ctr-by-position.png'))
display(Image(filename='paper/assets/ranked-review-cohorts.png'))

## Interpretation and action playbook

The random forest reduces pooled client-held-out weighted MAE only modestly versus the simple baseline. The fold-level scores vary; this is not strong evidence that the more complex model is universally preferable. The aggregate cohort rankings use only groups with at least 100 eligible pages and net-positive aggregate expected-minus-observed gaps. Those click-equivalent gaps are descriptive proxies, not recoverable-click estimates.

For page-one groups, review snippet-to-intent fit and competing result formats while protecting visibility. For lower-position groups, inspect relevance, coverage, and technical/indexing fundamentals before considering a CTR-only edit. Verify any action against authorized source systems and evaluate changes with controlled monitoring.

The outputs do not identify pages, establish the cause of observed CTR, forecast a future window, prove Google's ranking logic, or show causal refresh impact.